# Kimi

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"]     = "/root/autodl-tmp/LLM_Model"

import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

In [2]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

2026-04-03 08:37:10.951 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2026-04-03 08:37:10.953 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-03 08:37:10.954 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model
`torch_dtype` is deprecated! Use `dtype` instead!
using normal flash attention


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2026-04-03 08:37:18.094 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-03 08:37:18.095 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-04-03 08:37:18.983 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-04-03 08:37:19.176 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded.


In [3]:
SYSTEM_PROMPT = (
    "Reply one word: Dementia or Control."
)
USER_PROMPT = "Dementia or Control?"

In [4]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [5]:
VALID_LABELS = {"Dementia", "Healthy"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    messages = [
        {"role": "user", "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user", "message_type": "audio", "content": str(wav_path)},
    ]
    _, text = model.generate(messages, output_type="text")
    return text


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

In [6]:
OUTPUT_DIR = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result")


def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            pred = parse_prediction(raw)
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1)*100:.2f}%")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1)*100:.2f}%")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [7]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt_origin", "Pitt_origin", "Pitt_origin_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt_origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin-raw")

[Pitt-origin-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt_origin, exists=True


Pitt-origin-raw:   0%|          | 1/552 [00:01<17:14,  1.88s/it]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-origin-raw:   0%|          | 2/552 [00:02<09:28,  1.03s/it]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-raw:   1%|          | 3/552 [00:02<07:00,  1.30it/s]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-origin-raw: 100%|██████████| 552/552 [03:34<00:00,  2.57it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-origin-raw.csv
[Pitt-origin-raw]
  Accuracy:    60.69%
  F1:          0.7337
  Control Acc: 14.81%
  Dementia Acc:96.76%
  Valid: 552/552  Skipped: 0


In [8]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
# audio_dir = PROJECT_ROOT / "data/raw/Lu"
# evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:00<00:24,  2.95it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-raw:   3%|▎         | 2/74 [00:00<00:19,  3.62it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-raw:   4%|▍         | 3/74 [00:00<00:19,  3.69it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-raw: 100%|██████████| 74/74 [00:18<00:00,  4.11it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Lu-raw.csv
[Lu-raw]
  Accuracy:    47.30%
  F1:          0.6355
  Control Acc: 2.78%
  Dementia Acc:89.47%
  Valid: 74/74  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Demucs")

[Pitt-origin-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-Demucs, exists=True


Pitt-origin-Demucs:   0%|          | 1/552 [00:00<02:34,  3.57it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-origin-Demucs:   0%|          | 2/552 [00:00<02:47,  3.27it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


Pitt-origin-Demucs:   1%|          | 3/552 [00:00<02:46,  3.31it/s]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-origin-Demucs: 100%|██████████| 552/552 [02:41<00:00,  3.42it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-origin-Demucs.csv
[Pitt-origin-Demucs]
  Accuracy:    59.24%
  F1:          0.7286
  Control Acc: 10.29%
  Dementia Acc:97.73%
  Valid: 552/552  Skipped: 0


In [10]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
# evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [11]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Denoiser")

[Pitt-origin-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-Denoiser, exists=True


Pitt-origin-Denoiser:   0%|          | 1/552 [00:00<02:26,  3.77it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-origin-Denoiser:   0%|          | 2/552 [00:00<02:23,  3.84it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-Denoiser:   1%|          | 3/552 [00:00<02:21,  3.87it/s]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-origin-Denoiser: 100%|██████████| 552/552 [02:19<00:00,  3.96it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-origin-Denoiser.csv
[Pitt-origin-Denoiser]
  Accuracy:    57.43%
  F1:          0.7232
  Control Acc: 4.12%
  Dementia Acc:99.35%
  Valid: 552/552  Skipped: 0


In [12]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
# evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

In [13]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-origin-FRCRN_SE")

[Pitt-origin-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-FRCRN_SE, exists=True


Pitt-origin-FRCRN_SE:   0%|          | 1/552 [00:00<02:13,  4.14it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-FRCRN_SE:   0%|          | 2/552 [00:00<02:24,  3.80it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


Pitt-origin-FRCRN_SE:   1%|          | 3/552 [00:00<02:22,  3.86it/s]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-origin-FRCRN_SE: 100%|██████████| 552/552 [02:19<00:00,  3.95it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-origin-FRCRN_SE.csv
[Pitt-origin-FRCRN_SE]
  Accuracy:    58.15%
  F1:          0.7207
  Control Acc: 9.47%
  Dementia Acc:96.44%
  Valid: 552/552  Skipped: 0


In [14]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
# evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

In [15]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-origin-MossFormer")

[Pitt-origin-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-MossFormer, exists=True


Pitt-origin-MossFormer:   0%|          | 1/552 [00:00<02:14,  4.10it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-MossFormer:   0%|          | 2/552 [00:00<02:18,  3.97it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-MossFormer:   1%|          | 3/552 [00:00<02:19,  3.94it/s]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-origin-MossFormer: 100%|██████████| 552/552 [02:18<00:00,  3.97it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-origin-MossFormer.csv
[Pitt-origin-MossFormer]
  Accuracy:    58.15%
  F1:          0.7253
  Control Acc: 6.58%
  Dementia Acc:98.71%
  Valid: 552/552  Skipped: 0


In [16]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
# evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [17]:
csv       = PROJECT_ROOT / "data/processed/Pitt_origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-origin-Resemble")

[Pitt-origin-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-origin-Resemble, exists=True


Pitt-origin-Resemble:   0%|          | 1/552 [00:00<02:44,  3.34it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-origin-Resemble:   0%|          | 2/552 [00:00<02:52,  3.19it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-origin-Resemble:  74%|███████▍  | 410/552 [02:02<01:16,  1.87it/s]

  INVALID [409] session=268-0 true=Dementia raw='A man is talking about a picture.'


Pitt-origin-Resemble: 100%|██████████| 552/552 [02:43<00:00,  3.38it/s]

  Saved to /root/autodl-tmp/Few-Shot_is_all_you_need/LLM/kimi_audio_result/Pitt-origin-Resemble.csv
[Pitt-origin-Resemble]
  Accuracy:    57.35%
  F1:          0.7206
  Control Acc: 5.35%
  Dementia Acc:98.38%
  Valid: 551/552  Skipped: 0


In [18]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
# evaluate_dataset(csv, audio_dir, "Lu-Resemble")